# E008 — Bake-off des ControlNet QR SD1.5

Ce notebook compare les modèles sur exactement les mêmes prompts, payloads, seeds, paramètres SRPG et 26 validations. Aucun modèle n'est promu sur sa popularité ou sur des exemples choisis par son auteur.

In [ ]:
EXPERIMENT_NAME = "e008-controlnet-bakeoff-v1"
PROFILE_NAMES = (
    "dion_sd15",
    "monster_sd15_v1",
    "monster_sd15_v2",
    "nacholmo_sd15_v2",
)
CONTROL_SCALES = (0.90, 1.10, 1.35, 1.60)
CONTEXT_LIMIT = None  # mettre 2 uniquement pour un smoke test, puis changer EXPERIMENT_NAME


## 1. Porte GPU
Exécuter avant tout chargement CUDA. L'API QR et vLLM doivent avoir été arrêtés par le lanceur distant.

In [ ]:
from prooftag_qr.optimization import require_exclusive_gpu

require_exclusive_gpu()
print("GPU exclusif : OK")

## 2. Protocole apparié
Quatre modèles × quatre échelles × douze contextes = 192 exécutions complètes. Chaque exécution mesure séparément le brut ControlNet et la sortie SRPG à 100 pas.

In [ ]:
import csv
import gc
import json
import shutil
import traceback
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

from prooftag_qr.config import Settings
from prooftag_qr.controlnet_benchmark import (
    aggregate_controlnet_benchmark,
    best_trial_per_model,
    controlnet_trials,
    select_promotable_controlnet,
)
from prooftag_qr.controlnet_models import CONTROLNET_PROFILES
from prooftag_qr.optimization import E007Experiment, factorial_contexts

root = Path("/data/parameter-search") / EXPERIMENT_NAME
root.mkdir(parents=True, exist_ok=True)
contexts = list(factorial_contexts())[:CONTEXT_LIMIT]
profiles = [CONTROLNET_PROFILES[name] for name in PROFILE_NAMES]
print(f"{len(profiles)} modèles, {len(contexts)} contextes, {len(CONTROL_SCALES)} échelles")

## 3. Exécution séquentielle et reprenable
Un seul pipeline réside en VRAM. Les résultats et erreurs de chargement sont persistés avant de passer au modèle suivant.

In [ ]:
model_load_errors = []
for profile in profiles:
    print(f"\n===== {profile.name} =====")
    settings = Settings(
        data_dir=Path("/data"),
        model_cache_dir=Path("/cache"),
        default_backend="controlnet",
        controlnet_pipeline_mode="img2img",
        controlnet_model_id=profile.model_id,
        controlnet_model_subfolder=profile.subfolder,
        controlnet_conditioning_profile=profile.conditioning_profile,
        device="cuda",
        srpg_enabled=False,
        guided_rediffusion_enabled=False,
        latent_refinement_enabled=False,
    )
    try:
        experiment = E007Experiment(settings, f"{EXPERIMENT_NAME}/{profile.name}")
    except Exception as error:
        model_load_errors.append(
            {
                "profile": asdict(profile),
                "error": repr(error),
                "traceback": traceback.format_exc(),
            }
        )
        (root / "model-load-errors.json").write_text(
            json.dumps(model_load_errors, indent=2), encoding="utf-8"
        )
        gc.collect()
        torch.cuda.empty_cache()
        continue
    for context in contexts:
        for trial in controlnet_trials(profile, CONTROL_SCALES):
            row = experiment.execute("model-bakeoff", context, trial)
            print(
                f"{row['key']} raw={row.get('raw_passed', 0)}/26 "
                f"final={row.get('passed', 0)}/26"
            )
    pipeline = experiment.pipeline
    experiment.pipeline = None
    experiment.backend._pipeline = None
    pipeline.to("cpu")
    del pipeline, experiment
    gc.collect()
    torch.cuda.empty_cache()
    remaining_mib = torch.cuda.memory_allocated() / 1024**2
    print(f"VRAM PyTorch restante : {remaining_mib:.1f} MiB")
    if remaining_mib > 64:
        raise RuntimeError("Plus de 64 MiB restent alloués avant le prochain ControlNet")


## 4. Agrégation et classement strict
La complétude et le pire contexte précèdent la moyenne. CLIP-aesthetic et CLIPScore ne départagent qu'ensuite.

In [ ]:
all_rows = []
for profile in profiles:
    path = root / profile.name / "results.jsonl"
    if path.exists():
        all_rows.extend(
            json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line
        )
aggregates = aggregate_controlnet_benchmark(
    all_rows, expected_contexts=len(contexts)
)
(root / "controlnet-aggregates.json").write_text(
    json.dumps(aggregates, indent=2), encoding="utf-8"
)
for row in aggregates:
    print(
        f"{row['trial']:36s} raw={row['raw_mean_pass_rate']:.1%} "
        f"final={row['mean_pass_rate']:.1%} worst={row['worst_pass_rate']:.1%} "
        f"aes={row['mean_clip_aesthetic']:.3f}"
    )


## 5. Tableaux et graphiques
Les graphiques montrent le brut et le pipeline complet pour éviter d'attribuer à ControlNet un succès créé uniquement par SRPG.

In [ ]:
fields = sorted({key for row in aggregates for key in row})
with (root / "controlnet-aggregates.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(aggregates)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for row in aggregates:
    axes[0].scatter(
        row["raw_mean_pass_rate"],
        row["raw_mean_clip_aesthetic"],
        label=row["trial"],
    )
    axes[1].scatter(
        row["mean_pass_rate"],
        row["mean_clip_aesthetic"],
        label=row["trial"],
    )
axes[0].set(title="Stage-1 ControlNet seul", xlabel="Scan", ylabel="CLIP-aesthetic")
axes[1].set(title="Après SRPG 100 pas", xlabel="Scan", ylabel="CLIP-aesthetic")
for axis in axes:
    axis.axvline(1.0, color="red", linestyle="--")
    axis.grid(alpha=0.25)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
fig.tight_layout()
fig.savefig(root / "controlnet-raw-vs-srpg.png", dpi=160, bbox_inches="tight")
display(fig)


## 6. Porte de promotion et export
Un meilleur résultat observé n'est pas automatiquement un modèle de production. La promotion exige 26/26 sur tous les contextes.

In [ ]:
winner = select_promotable_controlnet(aggregates)
if winner is None:
    decision = {
        "status": "NO_PROMOTION",
        "reason": "aucun modèle complet strict sur tous les contextes",
        "best_observed": aggregates[0] if aggregates else None,
    }
    display(Markdown("## Aucune promotion : aucun profil strict sur tous les contextes."))
else:
    decision = {"status": "AUTOMATIC_CANDIDATE", "winner": winner}
    display(Markdown(f"## Candidat automatique : `{winner['trial']}`"))
physical_template = root / "physical-validation-template.csv"
physical_path = root / "physical-validation.csv"
candidates = best_trial_per_model(aggregates)[:3]
with physical_template.open("w", newline="", encoding="utf-8") as stream:
    fields = [
        "trial", "context_id", "image", "device", "scanner",
        "display_or_print", "distance_cm", "angle_deg", "lighting",
        "exact_payload_match", "latency_ms", "notes",
    ]
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    for candidate in candidates:
        for item in all_rows:
            if item.get("status") == "ok" and item["trial"] == candidate["trial"]:
                writer.writerow(
                    {
                        "trial": item["trial"],
                        "context_id": item["context_id"],
                        "image": item["image"],
                    }
                )
if not physical_path.exists():
    shutil.copyfile(physical_template, physical_path)
(root / "decision.json").write_text(
    json.dumps(decision, indent=2), encoding="utf-8"
)
archive = shutil.make_archive(
    str(Path("/workspace/results") / EXPERIMENT_NAME),
    "gztar",
    root_dir=root.parent,
    base_dir=EXPERIMENT_NAME,
)
print(f"Archive : {archive}")